# EDA und Datenvorbereitung

Ziel dieses Notebooks ist es, die Data-Collection-Outputs zu verstehen, erste Qualitätschecks durchzuführen und die Datasets für die Analyse vorzubereiten

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
DATA_DIR = Path("../../data")
INTERIM_DIR = DATA_DIR / "interim"

In [ ]:
COSTS_FILE = INTERIM_DIR / "gesundheitskosten_2011_2026.csv"

df_costs = pd.read_csv(COSTS_FILE)

print(pd.DataFrame({
        "rows": [len(df_costs)],
        "columns": [df_costs.shape[1]],
        "duplicate_rows": [df_costs.duplicated().sum()],
        "missing_cells": [df_costs.isna().sum().sum()],
}))
empty_columns = df_costs.columns[df_costs.isna().all()].tolist()
print(empty_columns)

Die Gesundheitskosten enthalten 44’232 fehlende Werte. Diese entstehen  durch sechs optionale Metadatenspalten die leer sind. Die analyse-relevanten Spalten wie Jahr, Kanton, Alter, Beobachtungswert, Multiplikator und Status enthalten keine fehlenden Werte.

Es sind ebenfalls viele Spalten vorhanden, welche für die Analyse später nicht relevant sind.

In [ ]:
cost_columns = [
    "TIME_PERIOD",
    "CANTON",
    "Swiss cantons",
    "AGE",
    "Age groups",
    "OBS_VALUE",
    "MULT",
    "OBS_STATUS",
    "Code list for Observation Status",
]

df_costs_processed = df_costs[cost_columns].copy()
df_costs_processed.head(30)

In [ ]:
print(pd.DataFrame({
        "year_min": [df_costs_processed["TIME_PERIOD"].min()],
        "year_max": [df_costs_processed["TIME_PERIOD"].max()],
        "n_years": [df_costs_processed["TIME_PERIOD"].nunique()],
        "n_cantons": [df_costs_processed["Swiss cantons"].nunique()],
        "n_age_groups": [df_costs_processed["Age groups"].nunique()],
        "n_status": [df_costs_processed["OBS_STATUS"].nunique()],
        "n_multipliers": [df_costs_processed["MULT"].nunique()],
}))

In [ ]:
print(f'Status: \n {df_costs_processed["OBS_STATUS"].unique()}')
print(f'Kantone: \n {df_costs_processed["Swiss cantons"].unique()}')

In [ ]:
pd.crosstab(
    df_costs_processed["TIME_PERIOD"],
    df_costs_processed["OBS_STATUS"]
)

Für den Status haben 3 verschiedene Werte:

- A: Normaler Wert
- E: Geschätzter Wert
- P: Provisorischer Wert

Das Jahr 2023 beinhaltet nur provisorische Werte für alle Zeilen.
Das Jahr 2024 beinhaltet insgesamt nur einen geschätzten Wert.

Die Spalte Kantone beinhaltet alle 26 Kantone und das Total für die Schweiz

Um den absolut Wert für die Kosten zu erhalten müssen wird eine zusätzliche Spalte einfügen

In [ ]:
df_costs_processed["costs_chf"] = df_costs["OBS_VALUE"] * (10 ** df_costs["MULT"])
df_costs_processed.head()

In [ ]:
df_costs_total_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] == "Total")
].copy()

px.line(
    df_costs_total_ch,
    x="TIME_PERIOD",
    y="costs_chf",
    markers=True,
    title="Gesundheitskosten Schweiz, Total",
    labels={"TIME_PERIOD": "Jahr", "costs_chf": "Kosten in CHF"},
)

In [ ]:
df_costs_canton_total = df_costs_processed[
    (df_costs_processed["Age groups"] == "Total") &
    (df_costs_processed["Swiss cantons"] != "Total")
].copy()

latest_canton_year = int(df_costs_canton_total["TIME_PERIOD"].max())

df_costs_canton_latest = (
    df_costs_canton_total[df_costs_canton_total["TIME_PERIOD"] == latest_canton_year]
    .sort_values("costs_chf", ascending=False)
)

px.bar(
    df_costs_canton_latest.head(10),
    x="Swiss cantons",
    y="costs_chf",
    title=f"Top 10 Kantone nach Gesundheitskosten {latest_canton_year}",
)

Im Plot für die Top 10 Kantone ist zu sehen, dass die Bevölkerungsgrösse eine Rolle spielt.

Für die spätere Analyse könnten die pro Kopf Kosten einen besseren Einblick geben

In [ ]:
df_costs_age_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] != "Total") &
    (df_costs_processed["TIME_PERIOD"] == 2023)
].copy()

df_costs_age_ch["cost_share"] = (
    df_costs_age_ch["costs_chf"] / df_costs_age_ch["costs_chf"].sum()
)

px.bar(
    df_costs_age_ch,
    x="Age groups",
    y="cost_share",
    title="Kostenanteil nach Altersgruppe Schweiz 2023",
)

In [ ]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

df_costs_processed.to_csv(PROCESSED_DIR / "gesundheitskosten.csv")